## Example 1: Using IPythonLexer to highlight IPython code

In [ ]:
from ipython_pygments_lexers import IPythonLexer
from pygments import highlight
from pygments.formatters import TerminalFormatter

# Example 1: Using IPythonLexer to highlight IPython code
print("Example 1: IPython code syntax highlighting")

# Sample IPython code with magic commands
ipython_code = """
%timeit x = sum(range(100))
!ls -la
df = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 6]})
result = df.head()
"""

# Create IPython lexer instance
lexer = IPythonLexer()
print(f"Lexer name: {lexer.name}")
print(f"Lexer aliases: {lexer.aliases}")

# Highlight the code
formatter = TerminalFormatter()
highlighted = highlight(ipython_code, lexer, formatter)
print("Highlighted IPython code:")
print(highlighted)

from pygments.lexers import get_lexer_by_name

assert lexer.name == "IPython" and "ipython3" in lexer.aliases
# The package registers its lexers with Pygments under these aliases
assert isinstance(get_lexer_by_name("ipython3"), IPythonLexer)
# Lexing never drops text: the tokens join back to the input
assert "".join(value for _, value in lexer.get_tokens(ipython_code)) == ipython_code.lstrip("\n")
assert "timeit" in highlighted and "DataFrame" in highlighted

## Example 2: Using IPythonConsoleLexer for console session highlighting

In [ ]:
from ipython_pygments_lexers import IPythonConsoleLexer
from pygments import highlight
from pygments.formatters import TerminalFormatter

# Example 2: Using IPythonConsoleLexer for console session highlighting
print("Example 2: IPython console session highlighting")

# Sample IPython console session
console_session = """
In [1]: import numpy as np

In [2]: arr = np.array([1, 2, 3, 4, 5])

In [3]: arr.mean()
Out[3]: 3.0

In [4]: %who
arr  np
"""

# Create IPython console lexer instance
lexer = IPythonConsoleLexer()
print(f"Console lexer name: {lexer.name}")
print(f"Console lexer aliases: {lexer.aliases}")

# Highlight the console session
formatter = TerminalFormatter()
highlighted = highlight(console_session, lexer, formatter)
print("Highlighted console session:")
print(highlighted)

from pygments.token import Generic

assert lexer.name == "IPython console session" and "ipythonconsole" in lexer.aliases
tokens = list(lexer.get_tokens(console_session))
prompts = [value for token_type, value in tokens if token_type in Generic.Prompt]
# Input prompts are recognised as prompts
assert prompts[:4] == ["In [1]: ", "In [2]: ", "In [3]: ", "In [4]: "]
assert any(token_type in Generic.Output and "3.0" in value for token_type, value in tokens)

## Example 3: Using IPythonTracebackLexer for error highlighting

In [ ]:
from ipython_pygments_lexers import IPythonTracebackLexer
from pygments import highlight
from pygments.formatters import TerminalFormatter

# Example 3: Using IPythonTracebackLexer for error highlighting
print("Example 3: IPython traceback highlighting")

# Sample IPython traceback
traceback_text = """
---------------------------------------------------------------------------
ZeroDivisionError                         Traceback (most recent call last)
<ipython-input-1-9e1622b385b6> in <module>
      1 x = 1
----> 2 y = x / 0
      3 print(y)

ZeroDivisionError: division by zero
"""

# Create IPython traceback lexer instance
lexer = IPythonTracebackLexer()
print(f"Traceback lexer name: {lexer.name}")
print(f"Traceback lexer aliases: {lexer.aliases}")

# Highlight the traceback
formatter = TerminalFormatter()
highlighted = highlight(traceback_text, lexer, formatter)
print("Highlighted traceback:")
print(highlighted)

from pygments.token import Generic, Name

assert lexer.name == "IPython Traceback" and "ipythontb" in lexer.aliases
tokens = list(lexer.get_tokens(traceback_text))
assert tokens[0] == (Generic.Traceback, "-" * 75 + "\n")
# The exception name is tagged as an exception, both in the header and on the last line
assert [value for token_type, value in tokens if token_type in Name.Exception and value == "ZeroDivisionError"] == ["ZeroDivisionError"] * 2

## Example 4: Using IPythonPartialTracebackLexer and IPython3Lexer

In [ ]:
from ipython_pygments_lexers import IPythonPartialTracebackLexer, IPython3Lexer
from pygments import highlight
from pygments.formatters import TerminalFormatter

# Example 4: Using IPythonPartialTracebackLexer and IPython3Lexer
print("Example 4: IPython partial traceback and Python3 lexers")

# Sample partial traceback
partial_traceback = """
      4 def divide_numbers(a, b):
----> 5     return a / b
      6 
      7 result = divide_numbers(10, 0)
"""

# Create partial traceback lexer
partial_lexer = IPythonPartialTracebackLexer()
print(f"Partial traceback lexer: {partial_lexer.name}")

# Create IPython3 lexer
python3_lexer = IPython3Lexer()
print(f"IPython3 lexer: {python3_lexer.name}")
print(f"IPython3 aliases: {python3_lexer.aliases}")

# Highlight partial traceback
formatter = TerminalFormatter()
highlighted_partial = highlight(partial_traceback, partial_lexer, formatter)
print("Highlighted partial traceback:")
print(highlighted_partial)

from pygments.token import Name, Number

assert partial_lexer.name == "IPython Partial Traceback"
assert python3_lexer.name == "IPython" and python3_lexer.aliases == ["ipython", "ipython3"]
tokens = list(partial_lexer.get_tokens(partial_traceback))
# The arrow marks the failing line, and line numbers are tagged as numbers
assert (Name.Exception, "----> ") in tokens
assert [value for token_type, value in tokens if token_type in Number.Integer] == ["4", "5", "6", "7"]

## Example 5: Token analysis with IPython code

In [ ]:
from ipython_pygments_lexers import IPythonLexer, ipython_tokens
from pygments.token import Token

# Example 5: Token analysis with IPython code
print("Example 5: IPython token analysis")

# Sample IPython code for tokenization
code_sample = """
%matplotlib inline
x = [1, 2, 3]
!echo "Hello World"
"""

# Create lexer and get tokens
lexer = IPythonLexer()
tokens = list(lexer.get_tokens(code_sample))

print("Token analysis:")
print(f"Total tokens: {len(tokens)}")

# Analyze token types
token_types = {}
for token_type, value in tokens:
    token_name = str(token_type)
    if token_name not in token_types:
        token_types[token_name] = 0
    token_types[token_name] += 1

print("Token type distribution:")
for token_type, count in sorted(token_types.items()):
    print(f"  {token_type}: {count}")

# Show special IPython tokens
print("Special IPython magic tokens:")
for token_type, value in tokens:
    if 'Magic' in str(token_type) or value.startswith('%') or value.startswith('!'):
        print(f"  {token_type}: '{value.strip()}'")

from pygments.token import Operator, Keyword, Name

# A line magic is a % operator followed by the magic name as a keyword
assert tokens[0] == (Operator, "%") and tokens[1] == (Keyword, "matplotlib")
# A shell escape is a ! operator followed by the command
bang = tokens.index((Operator, "!"))
assert tokens[bang + 1] == (Name.Builtin, "echo")
assert "".join(value for _, value in tokens) == code_sample.lstrip("\n")